# Bronze ingestion — Medidas (measurement metadata)

This notebook takes the raw `working_medidas` table, checks it for obvious data problems,
and saves an unchanged copy as `hive_metastore.bronze.bronze_medidas`.

This table is a reference list rather than telemetry: there are about 258k rows
and about 258k distinct `TAG` values, so roughly one row per `TAG`. That fits a metadata
table describing each measurement point (for example its operational limits), as opposed
to the ARQLMED table, which has many readings per point over time.

**Columns used below** (read from how they're used, not a documented list):
- `TAG` — the identifier, one per row.

| rows | distinct TAG |
|---|---|
| 258,378 | 258,007 |

## 1. Setup

Load the shared helper functions and imports from the project utilities notebook.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

## 2. Load the raw data

Read the source table. The preview and schema print-out confirm the columns and types
look right before going further.

In [0]:
df = spark.read.table("hive_metastore.default.working_medidas")
display(df.limit(5))
df.printSchema()

## 3. Data checks

A few checks over the whole table before saving it. None of these change the data.

### 3.1 Size

Total rows and number of distinct `TAG` values. The two numbers being almost equal is
what tells us this is close to one row per `TAG`.

In [0]:
summary = df.agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("TAG").alias("n_ids")
)
display(summary)

### 3.2 Empty values

Count missing values per column, then the same as a share of all rows.

In [0]:
n = df.count()

nulls = df.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
    for c in df.columns
])

nulls_pct = nulls.select(*[
    (F.col(c) / F.lit(n)).alias(c.replace("null_", "pct_null_"))
    for c in nulls.columns
])

display(nulls)
display(nulls_pct)

### 3.3 Zero values

For each column, count rows holding a plain `"0"` (ignoring spaces). As in the other
bronze notebooks, this matches only the exact text `"0"`, not `"0.0"` or similar.

In [0]:
agg_exprs = [
    F.sum((F.trim(F.col(c)) == F.lit("0")).cast("int")).alias(c)
    for c in df.columns
]

zero_counts = df.agg(*agg_exprs)
display(zero_counts)

Show the zero counts as a share of all rows.

In [0]:
zero_pct = zero_counts.select(*[
    (F.col(c) / F.lit(n)).alias(c)
    for c in zero_counts.columns
])

display(zero_pct)

## 4. Save to bronze

Save the table as Delta. `overwrite` plus `overwriteSchema` means you can re-run this
cell safely.

**Target table:** `hive_metastore.bronze.bronze_medidas`

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "bronze"
target_table = "bronze_medidas"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


Save the table and show the result.

> Minor: this cell reads with `spark.table(...)`, while cell 2 and the rest of the project
> use `spark.read.table(...)`. Same effect, just inconsistent.

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))
